# Energy Consumption Regression Analysis

This notebook analyzes the energy consumption of various energy types based on the average output tokens per prompt using polynomial regression models. The steps involve loading the data, transforming it, fitting the regression models, predicting values, and visualizing the results.


## 1. Load Data

First, we load the data from a CSV file into a pandas DataFrame.

In [14]:
import pandas as pd
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import numpy as np
import matplotlib.pyplot as plt

import altair as alt

from IPython.display import display, Math

In [15]:
# Function to load CSV data
def load_csv_data(file_path):
    """
    Load data from a CSV file into a pandas DataFrame.
    
    Parameters:
        file_path (str): The path to the CSV file.
    
    Returns:
        pd.DataFrame: The loaded DataFrame.
    """
    df = pd.read_csv(file_path)
    return df

In [16]:
# Specify the path to your CSV file
file_path = 'data/emission_regression_vllm.csv'

# Load the data into a DataFrame
df_vllm_emission_regression = load_csv_data(file_path)

# Display the first few rows of the DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_time,time_per_prompt,tok_per_sec,total_out_tok,total_in_tok,...,actual_ram_energy_per_10k_prompts,actual_idle_gpu_energy_per_10k_prompts,actual_non_idle_gpu_energy_per_10k_prompts,pred_emissions_per_10k_prompts,pred_total_energy_per_10k_prompts,pred_cpu_energy_per_10k_prompts,pred_gpu_energy_per_10k_prompts,pred_ram_energy_per_10k_prompts,pred_idle_gpu_energy_per_10k_prompts,pred_non_idle_gpu_energy_per_10k_prompts
0,Output-tok-vllm,llama3,8,1,1000,49.202580,0.049203,369.350551,18173.0,186660.0,...,0.011358,0.015307,0.018832,0.009760,0.014678,0.002586,0.007931,0.004162,0.005577,0.002354
1,Output-tok-vllm,llama3,8,2,1000,51.667090,0.051667,416.570781,21523.0,186660.0,...,0.011914,0.016074,0.022147,0.011703,0.017600,0.002954,0.009896,0.004751,0.006371,0.003525
2,Output-tok-vllm,llama3,8,3,1000,63.470966,0.063471,643.806179,40863.0,186660.0,...,0.014658,0.019747,0.027444,0.022920,0.034469,0.005077,0.021242,0.008151,0.010958,0.010284
3,Output-tok-vllm,llama3,8,5,1000,80.341208,0.080341,869.516431,69858.0,186660.0,...,0.018542,0.024995,0.032605,0.039737,0.059759,0.008260,0.038251,0.013247,0.017834,0.020417
4,Output-tok-vllm,llama3,8,10,1000,144.784709,0.144785,1258.675737,182237.0,186660.0,...,0.033415,0.045044,0.061875,0.104914,0.157778,0.020599,0.104178,0.033001,0.044486,0.059692
5,Output-tok-vllm,llama3,8,15,1000,211.358977,0.211359,1363.594793,288208.0,186660.0,...,0.048771,0.065756,0.089521,0.166375,0.250207,0.032234,0.166345,0.051629,0.069618,0.096727
6,Output-tok-vllm,llama3,8,20,1000,276.606541,0.276607,1439.821340,398264.0,186660.0,...,0.063828,0.086055,0.118312,0.230205,0.346200,0.044317,0.230909,0.070974,0.095719,0.135189
7,Output-tok-vllm,llama3,8,30,1000,408.543100,0.408543,1455.834157,594771.0,186660.0,...,0.094263,0.127102,0.175979,0.344175,0.517597,0.065893,0.346188,0.105516,0.142323,0.203865
8,Output-tok-vllm,llama3,8,40,1000,541.179589,0.541180,1443.934722,781428.0,186660.0,...,0.124856,0.168367,0.234142,0.452432,0.680402,0.086386,0.455689,0.138326,0.186591,0.269098
9,Output-tok-vllm,llama3,8,60,1000,882.308407,0.882308,1362.874920,1202476.0,186660.0,...,0.203499,0.274496,0.393358,0.696631,1.047647,0.132615,0.702694,0.212338,0.286447,0.416247


## 2. Data Transformation

Convert energy values from kilowatt-hours (kWh) to watt-hours (Wh) for better granularity, and calculate prompts per second.

In [17]:
# Transform energy values from kWh to Wh
df_vllm_emission_regression['total_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_total_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['ram_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_ram_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['gpu_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_gpu_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['cpu_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_cpu_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['gpu_idle_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_idle_gpu_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['gpu_non_idle_energy_10k_prompts_Wh'] = df_vllm_emission_regression['actual_non_idle_gpu_energy_per_10k_prompts'] * 1000
df_vllm_emission_regression['prompt_per_sec'] = df_vllm_emission_regression['num_prompts'] / df_vllm_emission_regression['total_time']

df_vllm_emission_regression = df_vllm_emission_regression[['test_type', 
                                                           'model_type', 
                                                           'parameters',
                                                           'num_examples', 
                                                           'num_prompts', 
                                                           'total_time', 
                                                           'prompt_per_sec', 
                                                           'total_out_tok', 
                                                           'total_in_tok', 
                                                           'avg_out_tok', 
                                                           'avg_in_tok', 
                                                           'total_energy_10k_prompts_Wh', 
                                                           'ram_energy_10k_prompts_Wh', 
                                                           'gpu_energy_10k_prompts_Wh', 
                                                           'cpu_energy_10k_prompts_Wh',
                                                           'gpu_idle_energy_10k_prompts_Wh', 
                                                           'gpu_non_idle_energy_10k_prompts_Wh']]

# Display the updated DataFrame
df_vllm_emission_regression

,test_type,model_type,parameters,num_examples,num_prompts,total_time,prompt_per_sec,total_out_tok,total_in_tok,avg_out_tok,avg_in_tok,total_energy_10k_prompts_Wh,ram_energy_10k_prompts_Wh,gpu_energy_10k_prompts_Wh,cpu_energy_10k_prompts_Wh,gpu_idle_energy_10k_prompts_Wh,gpu_non_idle_energy_10k_prompts_Wh
0,Output-tok-vllm,llama3,8,1,1000,49.202580,20.324137,18173.0,186660.0,18.173,186.66,52.586817,11.357943,34.139819,7.089055,15.307469,18.832350
1,Output-tok-vllm,llama3,8,2,1000,51.667090,19.354680,21523.0,186660.0,21.523,186.66,57.581257,11.913573,38.220994,7.446690,16.074206,22.146788
2,Output-tok-vllm,llama3,8,3,1000,63.470966,15.755235,40863.0,186660.0,40.863,186.66,70.997913,14.658208,47.190034,9.149671,19.746523,27.443512
3,Output-tok-vllm,llama3,8,5,1000,80.341208,12.446913,69858.0,186660.0,69.858,186.66,87.716061,18.541585,57.599581,11.574894,24.995042,32.604538
4,Output-tok-vllm,llama3,8,10,1000,144.784709,6.906807,182237.0,186660.0,182.237,186.66,161.191220,33.415311,106.918879,20.857030,45.044132,61.874748
5,Output-tok-vllm,llama3,8,15,1000,211.358977,4.731287,288208.0,186660.0,288.208,186.66,234.492818,48.770591,155.277044,30.445183,65.756126,89.520918
6,Output-tok-vllm,llama3,8,20,1000,276.606541,3.615244,398264.0,186660.0,398.264,186.66,308.037495,63.828095,204.366979,39.842421,86.055368,118.311611
7,Output-tok-vllm,llama3,8,30,1000,408.543100,2.447722,594771.0,186660.0,594.771,186.66,456.190371,94.263391,303.081193,58.845788,127.102298,175.978895
8,Output-tok-vllm,llama3,8,40,1000,541.179589,1.847815,781428.0,186660.0,781.428,186.66,605.313415,124.856061,402.508619,77.948735,168.366983,234.141635
9,Output-tok-vllm,llama3,8,60,1000,882.308407,1.133391,1202476.0,186660.0,1202.476,186.66,998.433875,203.499210,667.853812,127.080853,274.495949,393.357863


## 3. Polynomial Regression Model Fitting
Fit polynomial regression models for each type of energy consumption using the average output tokens per prompt as the feature variable.

In [18]:
def fit_polynomial_regression(X, y, degree=2):
    polynomial_features = PolynomialFeatures(degree=degree)
    linear_regression = LinearRegression()
    model = make_pipeline(polynomial_features, linear_regression)
    model.fit(X, y)
    return model

In [19]:
# Define the feature variable
X = df_vllm_emission_regression[['avg_out_tok']]

In [20]:
# Fit models for each energy consumption type
models = {}
energy_types = [
    'total_energy_10k_prompts_Wh', 
    'ram_energy_10k_prompts_Wh', 
    'gpu_energy_10k_prompts_Wh', 
    'cpu_energy_10k_prompts_Wh',
    'gpu_idle_energy_10k_prompts_Wh', 
    'gpu_non_idle_energy_10k_prompts_Wh'
]

In [21]:
for energy_type in energy_types:
    y = df_vllm_emission_regression[energy_type]
    models[energy_type] = fit_polynomial_regression(X, y)
    coefs = models[energy_type].named_steps['linearregression'].coef_
    intercept = models[energy_type].named_steps['linearregression'].intercept_
    

## 4. Display Model Coefficients
Display the coefficients of the polynomial regression models for each type of energy consumption.

In [22]:
def display_model_coefficients(model, energy_type):
    coefs = model.named_steps['linearregression'].coef_
    intercept = model.named_steps['linearregression'].intercept_

    # Format the coefficients to 4 decimal places for readability
    coefs = np.round(coefs, 5)
    intercept = np.round(intercept, 5)
    
    
    print("="*20 + f" Regression for {energy_type} " + "="*20 + "\n")
    
    # Print raw coefficients to check their values
    print(f"Raw coefficients:\n intercept={intercept}, coefs={coefs}\n")
    
    print("Formula:")
    # Generate the LaTeX formula
    latex_formula = (
        f"\\hat{{y}} = {intercept:.5f} + {coefs[1]:.5f} x + {coefs[2]:.5f} x^2"
    )

    # Display the LaTeX formula
    display(Math(latex_formula))

    print("\n\n")

In [23]:
# Display coefficients for each energy consumption type
for energy_type in energy_types:
    display_model_coefficients(models[energy_type], energy_type)

==================== Regression for total_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=46.46332, coefs=[0.0000e+00 5.8335e-01 1.7000e-04]

Formula:


<IPython.core.display.Math object>




==================== Regression for ram_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=9.62719, coefs=[0.0000e+00 1.2327e-01 3.0000e-05]

Formula:


<IPython.core.display.Math object>




==================== Regression for gpu_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=30.82249, coefs=[0.0000e+00 3.8318e-01 1.2000e-04]

Formula:


...

<IPython.core.display.Math object>




==================== Regression for gpu_idle_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=12.97939, coefs=[0.0000e+00 1.6613e-01 4.0000e-05]

Formula:


<IPython.core.display.Math object>




==================== Regression for gpu_non_idle_energy_10k_prompts_Wh ====================

Raw coefficients:
 intercept=17.8431, coefs=[0.0000e+00 2.1704e-01 8.0000e-05]

Formula:


<IPython.core.display.Math object>

## 5. Predict Values
Define x values for prediction and predict the corresponding energy consumption values using the fitted models.

In [24]:
# Define the x values for prediction
predicted_values = {'avg_out_tok': [10, 50, 250, 500, 1000, 1500, 2000]}

x_values = pd.DataFrame(predicted_values)

# Predict values ensuring feature names are consistent
for energy_type in energy_types:
    model = models[energy_type]
    predicted_values[energy_type] = model.predict(x_values)

# Display the predicted values
predicted_values_df = pd.DataFrame(predicted_values)
predicted_values_df

,avg_out_tok,total_energy_10k_prompts_Wh,ram_energy_10k_prompts_Wh,gpu_energy_10k_prompts_Wh,cpu_energy_10k_prompts_Wh,gpu_idle_energy_10k_prompts_Wh,gpu_non_idle_energy_10k_prompts_Wh
0,10,52.314314,10.863040,34.666572,6.784702,14.645027,20.021545
1,50,76.067726,15.869961,50.289028,9.908737,21.393507,28.895522
2,250,203.220979,42.429068,134.308258,26.483653,57.197925,77.110333
3,500,381.817688,79.200994,253.176713,49.439980,106.786310,146.390403
4,1000,804.528244,164.654991,537.061680,102.811573,222.072618,314.989062
5,1500,1314.594987,265.989180,882.477389,166.128418,358.838311,523.639078
6,2000,1912.017917,383.203562,1289.423841,239.390514,517.083389,772.340452


## 6. Combine Actual and Predicted Data
Combine the actual and predicted data into a single DataFrame for visualization.

In [25]:
# Combine actual and predicted data into a single DataFrame
data = []
for energy_type in energy_types:
    for index, row in df_vllm_emission_regression.iterrows():
        data.append({'avg_out_tok': row['avg_out_tok'], 'Energy_Consumption': row[energy_type], 'Type': 'Actual', 'Energy_Type': energy_type})
    for i, x in enumerate(x_values['avg_out_tok']):
        data.append({'avg_out_tok': x, 'Energy_Consumption': predicted_values[energy_type][i], 'Type': 'Predicted', 'Energy_Type': energy_type})

combined_df = pd.DataFrame(data)
combined_df

,avg_out_tok,Energy_Consumption,Type,Energy_Type
0,18.173,52.586817,Actual,total_energy_10k_prompts_Wh
1,21.523,57.581257,Actual,total_energy_10k_prompts_Wh
2,40.863,70.997913,Actual,total_energy_10k_prompts_Wh
3,69.858,87.716061,Actual,total_energy_10k_prompts_Wh
4,182.237,161.191220,Actual,total_energy_10k_prompts_Wh
...,...,...,...,...
103,250.000,77.110333,Predicted,gpu_non_idle_energy_10k_prompts_Wh
104,500.000,146.390403,Predicted,gpu_non_idle_energy_10k_prompts_Wh
105,1000.000,314.989062,Predicted,gpu_non_idle_energy_10k_prompts_Wh
106,1500.000,523.639078,Predicted,gpu_non_idle_energy_10k_prompts_Wh


## 7. Visualize Results
Create an Altair chart to visualize the actual and predicted energy consumption values.

In [26]:
# Create Altair chart
base = alt.Chart(combined_df[combined_df['Type'] == 'Actual']).mark_point(size=100, filled=True).encode(
    x=alt.X('avg_out_tok', title='Average Output Tokens per Prompt'),
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_out_tok', 'Energy_Consumption', 'Energy_Type', 'Type']
).properties(
    width=1200,
    height=600
)


# Highlight predicted values
predicted = alt.Chart(combined_df[combined_df['Type'] == 'Predicted']).mark_point(size=10, filled=False).encode(
    x=alt.X('avg_out_tok', title='Average Output Tokens per Prompt'), 
    y=alt.Y('Energy_Consumption', title='Energy Consumption (Wh)'),
    color=alt.Color('Energy_Type', title='Energy Type'),
    tooltip=['avg_out_tok', 'Energy_Consumption', 'Energy_Type']
)

regression = predicted.transform_regression('avg_out_tok', 'Energy_Consumption', groupby=['Energy_Type'], method="quad").mark_line()

# Combine charts
final_chart = base + regression + predicted

# Display the chart in Streamlit
final_chart

alt.LayerChart(...)